# Практична робота №8 — Нелінійне програмування з обмеженнями: умови ККТ, штрафні та бар'єрні функції

Вкажи номер варіанта, коли клітинка попросить — решта коду підхопить твою функцію, обмеження та послідовність r автоматично.

In [ ]:
import numpy as np
import sympy as sp
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
from variants import VARIANTS

_default = globals().get("VARIANT", 1)
_raw = input(f"Номер варіанта (1-{len(VARIANTS)}) [{_default}]: ").strip()
VARIANT = int(_raw) if _raw else _default
assert VARIANT in VARIANTS, f"Немає варіанта {VARIANT}"

## 1. Побудова моделі

f(x1,x2) — цільова функція, g1(x)<=0 і g2(x)<=0 — обмеження-нерівності, R_LIST — задана послідовність штрафних коефіцієнтів r (для бар'єрного методу використовується mu = 1/r).

In [ ]:
f_str, g1_str, g2_str, R_LIST = VARIANTS[VARIANT]

x1, x2 = sp.symbols('x1 x2', real=True)
f_sym = sp.sympify(f_str, locals={'x1': x1, 'x2': x2})
g1_sym = sp.sympify(g1_str, locals={'x1': x1, 'x2': x2})
g2_sym = sp.sympify(g2_str, locals={'x1': x1, 'x2': x2})

f_np = sp.lambdify((x1, x2), f_sym, 'numpy')
g1_np = sp.lambdify((x1, x2), g1_sym, 'numpy')
g2_np = sp.lambdify((x1, x2), g2_sym, 'numpy')

def f(x): return float(f_np(x[0], x[1]))
def g1(x): return float(g1_np(x[0], x[1]))
def g2(x): return float(g2_np(x[0], x[1]))
G = [g1, g2]

print(f"Варіант {VARIANT}")
print(f"f(x1,x2) = {f_sym}  ->  min")
print(f"g1: {g1_sym} <= 0,  g2: {g2_sym} <= 0")
print(f"R_LIST = {R_LIST}")

## 2. Безумовний мінімум — чи він допустимий?

Якщо безумовний мінімум f(x) уже задовольняє обидва обмеження — він одразу є розв'язком задачі. Інакше справжній оптимум лежить на межі допустимої області.

In [ ]:
grad_f_sym = [sp.diff(f_sym, v) for v in (x1, x2)]
unc_sol = sp.solve([sp.Eq(gi, 0) for gi in grad_f_sym], [x1, x2], dict=True)[0]
x_unc = (float(unc_sol[x1]), float(unc_sol[x2]))
print(f"безумовний мінімум: x = {x_unc}")
print(f"g1 = {g1(x_unc):+.4f}, g2 = {g2(x_unc):+.4f}")
print("допустимо" if g1(x_unc) <= 0 and g2(x_unc) <= 0 else "НЕДОПУСТИМО — оптимум на межі")

## 3. Аналітичний розв'язок за умовами ККТ

Перебираємо всі набори активних обмежень (жодне / g1 / g2 / обидва), для кожного розв'язуємо систему стаціонарності разом з рівняннями активних обмежень і перевіряємо допустимість та знак множників λ>=0.

In [ ]:
G_sym = [g1_sym, g2_sym]
grad_g_sym = [[sp.diff(g, v) for v in (x1, x2)] for g in G_sym]

def solve_kkt_case(active):
    if not active:
        x_try = x_unc
        ok = g1(x_try) <= 1e-7 and g2(x_try) <= 1e-7
        return dict(x=x_try, lam={0: 0.0, 1: 0.0}, accepted=ok)
    lam_syms = {i: sp.symbols(f'lam{i}', real=True) for i in active}
    stat_eqs = [sp.Eq(grad_f_sym[k] + sum(lam_syms[i] * grad_g_sym[i][k] for i in active), 0)
                for k in range(2)]
    act_eqs = [sp.Eq(G_sym[i], 0) for i in active]
    unknowns = [x1, x2] + [lam_syms[i] for i in active]
    sols = sp.solve(stat_eqs + act_eqs, unknowns, dict=True)
    if not sols:
        return dict(x=None, lam=None, accepted=False)
    sol = sols[0]
    x_try = (float(sol[x1]), float(sol[x2]))
    lam_val = {i: float(sol[lam_syms[i]]) for i in active}
    for i in (0, 1):
        lam_val.setdefault(i, 0.0)
    feasible = g1(x_try) <= 1e-6 and g2(x_try) <= 1e-6
    nonneg = all(lam_val[i] >= -1e-6 for i in active)
    return dict(x=x_try, lam=lam_val, accepted=feasible and nonneg)

for active in ([], [0], [1], [0, 1]):
    r = solve_kkt_case(active)
    status = "ПРИЙНЯТО" if r["accepted"] else "відкинуто"
    print(f"активні {active}: x={r['x']}, lam={r['lam']} -> {status}")
    if r["accepted"]:
        x_kkt, lam_kkt, active_case = r["x"], r["lam"], active

f_kkt = f(x_kkt)
print(f"\nx* = {x_kkt}, f* = {f_kkt}, lambda* = {lam_kkt}")

## 4. Метод зовнішніх штрафних функцій

P(x,r) = f(x) + r*sum(max(0,gi(x))^2) -> min, r зростає за заданою послідовністю R_LIST. Наближення йдуть ззовні допустимої області.

In [ ]:
def penalty(x, r):
    return f(x) + r * sum(max(0.0, g(x)) ** 2 for g in G)

rows = []
x_pen = np.array([0.0, 0.0])
for r in R_LIST:
    res = minimize(penalty, x_pen, args=(r,), method='BFGS')
    x_pen = res.x
    rows.append(dict(r=r, x1=x_pen[0], x2=x_pen[1], f=f(x_pen), g1=g1(x_pen)))
pd.DataFrame(rows)

## 5. Метод бар'єрних функцій

B(x,mu) = f(x) - mu*sum(ln(-gi(x))) -> min, mu = 1/r спадає. Старт — строго допустима точка; наближення йдуть зсередини.

In [ ]:
def barrier(x, mu):
    if any(g(x) >= -1e-12 for g in G):
        return 1e12
    return f(x) - mu * sum(np.log(-g(x)) for g in G)

x_bar = 0.3 * np.array(x_kkt)  # точка між 0 і x* — як правило строго всередині області
assert g1(x_bar) < 0 and g2(x_bar) < 0, "стартова точка має бути строго допустимою"

rows_b = []
for mu in [1.0 / r for r in R_LIST]:
    res = minimize(barrier, x_bar, args=(mu,), method='Nelder-Mead',
                    options={'xatol': 1e-10, 'fatol': 1e-12, 'maxiter': 20000})
    x_bar = res.x
    rows_b.append(dict(mu=mu, x1=x_bar[0], x2=x_bar[1], f=f(x_bar), g1=g1(x_bar)))
pd.DataFrame(rows_b)

## 6. Графік: допустима область і обидві траєкторії

In [ ]:
path_pen = np.array([[0.0, 0.0]] + [[r['x1'], r['x2']] for r in rows])
path_bar = np.array([list(0.3*np.array(x_kkt))] + [[r['x1'], r['x2']] for r in rows_b])
all_pts = np.vstack([path_pen, path_bar, [x_unc], [x_kkt]])
center = all_pts.mean(axis=0)
half_width = max(2.5, np.abs(all_pts - center).max() + 1.0)
xs = np.linspace(center[0]-half_width, center[0]+half_width, 300)
ys = np.linspace(center[1]-half_width, center[1]+half_width, 300)
X, Y = np.meshgrid(xs, ys)
Z = f_np(X, Y)
infeasible = (g1_np(X, Y) > 0) | (g2_np(X, Y) > 0)
levels = np.unique(np.quantile(Z, np.linspace(0, 1, 24)))

fig, ax = plt.subplots(figsize=(7, 6))
ax.contour(X, Y, Z, levels=levels, cmap='viridis', alpha=.7)
ax.contourf(X, Y, infeasible.astype(float), levels=[0.5, 1.5], colors=['0.75'], alpha=.5)
ax.plot(path_pen[:,0], path_pen[:,1], 'o-', color='crimson', label='штрафний (ззовні)')
ax.plot(path_bar[:,0], path_bar[:,1], 's-', color='royalblue', label="бар'єрний (зсередини)")
ax.plot(*x_kkt, 'g*', ms=16, label='x* (ККТ)')
ax.legend(fontsize=8); ax.set_xlabel('x1'); ax.set_ylabel('x2')
ax.set_title(f'Варіант {VARIANT}: допустима область і траєкторії')
fig.tight_layout()
fig

## 7. Метод SLSQP (пряме розв'язання з обмеженнями)

In [ ]:
cons = [{'type': 'ineq', 'fun': lambda x, g=g: -g(x)} for g in G]
res_slsqp = minimize(f, [0.0, 0.0], method='SLSQP', constraints=cons, options={'ftol': 1e-12})
print(f"SLSQP: x* = {res_slsqp.x}, f* = {res_slsqp.fun}, ітерацій = {res_slsqp.nit}")

## 8. Чисельна перевірка умов ККТ

In [ ]:
x_s = res_slsqp.x
grad_f_val = np.array([float(sp.lambdify((x1, x2), e, 'numpy')(*x_s)) for e in grad_f_sym])
A = np.array([[float(sp.lambdify((x1, x2), e, 'numpy')(*x_s)) for e in row] for row in grad_g_sym])
active_idx = [i for i, g in enumerate(G) if abs(g(x_s)) < 1e-6]
lam_num = np.zeros(2)
if active_idx:
    lam_num[active_idx] = np.linalg.lstsq(A[active_idx].T, -grad_f_val, rcond=None)[0]
print('стаціонарність (має бути ~0):', grad_f_val + A.T @ lam_num)
print('g1(x*), g2(x*):', g1(x_s), g2(x_s))
print('lambda:', lam_num, '>= 0:', all(lam_num >= -1e-9))
print('доп. нежорсткість lam_i*g_i:', [lam_num[i]*G[i](x_s) for i in range(2)])

## 9. Аналіз чутливості до правої частини активного обмеження

Теорія: lambda* = -d f*/d b, тому для малого зсуву Δb прогноз f*(b+Δ) ≈ f*(b) - lambda*·Δ. Перевіряємо на малому (Δ=0.01) і великому (Δ=1) зсуві — великий зсув може змінити набір активних обмежень, тому лінійний прогноз там природно менш точний.

In [ ]:
active_i = active_case[0] if active_case else None
if active_i is not None:
    g_expr_str = [g1_str, g2_str][active_i]
    rows_s = []
    for delta in (-1, -0.01, 0, 0.01, 1):
        shifted = sp.sympify(f"({g_expr_str}) - ({delta})", locals={'x1': x1, 'x2': x2})
        g_shift_np = sp.lambdify((x1, x2), shifted, 'numpy')
        Gd = [g1, g2]
        Gd[active_i] = lambda x, gf=g_shift_np: float(gf(x[0], x[1]))
        cons_d = [{'type': 'ineq', 'fun': lambda x, g=g: -g(x)} for g in Gd]
        rd = minimize(f, [0.0, 0.0], method='SLSQP', constraints=cons_d, options={'ftol': 1e-12})
        predicted = f_kkt - lam_kkt[active_i] * delta if delta != 0 else f_kkt
        rows_s.append(dict(delta=delta, f_actual=rd.fun, f_predicted=predicted,
                            diff=abs(rd.fun - predicted)))
    pd.DataFrame(rows_s)